In [0]:
from pyspark.sql import functions as F

In [0]:
encounters = spark.sql(
    """
    SELECT ENCOUNTER_NO
    FROM datahub_dev_bronze.datahub_msx.msx_ip_output
    WHERE ADMIT_DT_SRC > '2024-06-01'
       OR DSCH_DT_SRC > '2024-06-01'
    """
)

In [0]:
oracle_jdbc_url = dbutils.secrets.get(
    scope="oao_secrets",
    key="ORACLE_JDBC_URL",
)

oracle_password = dbutils.secrets.get(
    scope="oao_secrets",
    key="OAO_PRODUCTION",
)

oe_charge_detail = (
    spark.read
    .format("jdbc")
    .option("url", oracle_jdbc_url)
    .option(
        "dbtable",
        "OAO_PRODUCTION.OE_CHARGE_DETAIL",
    )
    .option("user", "OAO_PRODUCTION")
    .option("password", oracle_password)
    .option("driver", "oracle.jdbc.OracleDriver")
    .option("oracle.net.ssl_server_dn_match", "true")
    .option("fetchsize", "10000")
    .load()
)

In [0]:

charges = oe_charge_detail.withColumn(
    "HSP_ACCOUNT_ID",
    F.col("HSP_ACCOUNT_ID").cast("string"),
)

encounter_ids = encounters.select(
    F.col("ENCOUNTER_NO").cast("string").alias("HSP_ACCOUNT_ID")
)

filtered_oe_charge_detail = charges.join(
    encounter_ids,
    on="HSP_ACCOUNT_ID",
    how="left_semi",
)



In [0]:
selected_columns = ["HSP_ACCOUNT_ID", "FACILITY_ABBR", "COST_CENTER_C", "CHARGE_C", "EPIC_DEPT_ID", "EPIC_DEPT_NAME", "QUANTITY", "SERVICE_DATE", "CPT_HCPCS_C", "BILLING_CAT_C", "BILLING_CAT_DESC", "RPT_GRP_NINETEEN_C", "RPT_GRP_NINETEEN_DESC", "NEW_COST_CENTER_C", "NEW_GL_COMPONENT"]

formated_oe_charge_detail = filtered_oe_charge_detail.select(*selected_columns)

(
    formated_oe_charge_detail.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "opsanalytics_adb_workspace01.capacity_modeling.oe_charge_detail"
    )
)